# Phase 3 — Multi-Teacher Knowledge Distillation

Three fine-tuned teachers are fused in **logit space** and distilled into small students, alongside ground-truth supervision.

```
                                Knee X-ray
                                     |
        +----------------------------+----------------------------+
        |                            |                            |
        v                            v                            v
  Custom-ENB5                InceptionResNetV2                 ResNet50
   Teacher 1                    Teacher 2                      Teacher 3
        |                            |                            |
        v                            v                            v
    Logits z1                    Logits z2                    Logits z3
        |                            |                            |
        +----------------------------+----------------------------+
                                     |
                                     v
                          Multi-Teacher Fusion
                            zF = w1z1 + w2z2 + w3z3
                                     |
                                     v
                             Temperature T
                                     |
                                     v
                        Soft Targets  PT = softmax(zF / T)
                                     |
                                     v
                          Student (B0 / MobileNetV2 / B2)
                                     |
                                     v
                        Student soft  PS = softmax(zS / T)
                                     |
                                     v
                                  KL 0-4
```

At the same time the student sees the **ground-truth KL label**, so there are two sources of supervision.

### The maths implemented
| Quantity | Definition |
|---|---|
| Teacher fusion | $z_F=w_1z_1+w_2z_2+w_3z_3$, with $\sum w_i=1$ (equal weights, $w_i=1/3$) |
| Teacher soft targets | $P_T(c)=\dfrac{\exp(z_{F,c}/T)}{\sum_j \exp(z_{F,j}/T)}$ |
| Student soft targets | $P_S(c)=\dfrac{\exp(z_{S,c}/T)}{\sum_j \exp(z_{S,j}/T)}$ |
| Hard-label loss | $L_{CE}=-\sum_c y_c\log \mathrm{softmax}(z_S)_c$ |
| Distillation loss | $L_{KD}=T^2\,\mathrm{KL}(P_T\parallel P_S)$ |
| Total | $L_{Total}=\alpha L_{CE}+(1-\alpha)L_{KD}$ |

Fusion is in **logit** space, as in the diagram: averaging logits preserves each teacher's confidence structure, and the $T^2$ factor keeps the distillation gradients on the same scale as the hard-label ones.

### Teachers (from the phase-2 grid search, at their winning settings)
| Key | Model | Test macro F1 | Params | Winning config |
|---|---|---:|---:|---|
| `ENB5` | Custom-ENB5 | 0.883 | 32.2M | rmsprop, lr 1e-4, dropout 0.3, batch 32 |
| `IRV2` | InceptionResNetV2 | 0.877 | 57.5M | adam, lr 1e-4, dropout 0.5, batch 16 |
| `R50` | ResNet50 | 0.875 | 27.3M | adam, lr 1e-4, dropout 0.3, batch 32 |

Teacher weights are loaded from `../tuning/results/...` when present; otherwise each teacher is retrained from its recorded winning configuration and re-evaluated, so the reported teacher numbers always match the weights actually used. Each teacher keeps **its own input normalisation** (caffe for ResNet50, tf for InceptionResNetV2, pass-through for EfficientNet), so the generators yield raw augmented images and every network applies its own `preprocess_input`.

### Students and configurations
Students: **EfficientNetB0** (4.4M), **MobileNetV2** (2.6M), **EfficientNetB2** (8.1M) — backbone fully fine-tuned, with the compact KD head `GAP -> Dense(256, swish) -> Dropout(0.3) -> Dense(5)` producing logits.

Each student is trained under 8 configurations:

| Key | Teachers | Purpose |
|---|---|---|
| `none` | — | Control: hard labels only, no distillation |
| `ENB5`, `IRV2`, `R50` | one | Single-teacher distillation |
| `ENB5+IRV2`, `ENB5+R50`, `IRV2+R50` | two | Does a second teacher help? |
| `ENB5+IRV2+R50` | three | The fusion above |

All at T=3, alpha=0.3 (your earlier KD setting) so they are directly comparable, followed by a 3x3 sweep (T in 2/3/5, alpha in 0.1/0.3/0.5) on the best student+configuration.

### Protocol
Up to **300 epochs** with `EarlyStopping(val_loss, patience=10, restore_best_weights=True)`, Adam at lr 1e-4, batch 32, 224x224, the same augmentation as every earlier notebook. Training loss is the combined objective; **validation loss is hard-label cross-entropy**, so early stopping and model selection are comparable across all configurations, including the control.

### Outputs — `results/` next to this notebook
```
results/teachers/<Teacher>/       metrics.json, figures/, and the model used (teacher.keras)
results/teachers/_ensemble/       the fused 3-teacher ensemble evaluated on test (the ceiling the student chases)
results/<Student>/<config>/       metrics.json, history.csv, classification_report.*, confusion_matrix.csv,
                                  test_predictions.csv, figures/*.{pdf,png}, student.keras
results/_summary/                 summary.csv, advisor_table.csv, sweep.csv,
                                  figures/performance_vs_size.*, test_metrics.*, teacher_vs_student.*,
                                  training_time.*, val_loss_all.*
```

### How to run
1. Open with the TensorFlow GPU kernel. Section 1.2 installs anything missing.
2. Check `DATA_DIR` in section 1.1.
3. *Run All.* Every training step runs in a **fresh subprocess** and each finished run is skipped on a re-run, so the job is resumable and one crash cannot end it.
4. Budget roughly **8-10 hours**: 24 distillation runs (~15-20 min each, since three teachers run a forward pass on every batch) plus the sweep. Set `STUDENTS` in section 1.1 to run them one at a time.

## 1. Environment & Configuration
### 1.1 Settings

In [ ]:
import os

os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")


def _env(key, default, cast=str):
    value = os.environ.get(key)
    return default if value in (None, "") else cast(value)


def _env_list(key, default):
    value = os.environ.get(key)
    return default if value in (None, "") else [v.strip() for v in value.split(",") if v.strip()]


CFG = dict(
    DATA_DIR=_env("OSTEO_DATA_DIR", "/mnt/d/FA019/new/SplittedDataset"),
    OUTPUT_DIR=_env("OSTEO_OUTPUT_DIR", "results"),
    TUNING_DIR=_env("OSTEO_TUNING_DIR", "../tuning/results"),   # where phase-2 teachers live
    STUDENTS=_env_list("OSTEO_STUDENTS", ["EfficientNetB0", "MobileNetV2", "EfficientNetB2"]),
    CONFIGS=_env_list("OSTEO_CONFIGS", ["none", "ENB5", "IRV2", "R50",
                                        "ENB5+IRV2", "ENB5+R50", "IRV2+R50", "ENB5+IRV2+R50"]),
    TEMPERATURE=_env("OSTEO_T", 3.0, float),
    ALPHA=_env("OSTEO_ALPHA", 0.3, float),
    SWEEP_T=[float(v) for v in _env_list("OSTEO_SWEEP_T", ["2", "3", "5"])],
    SWEEP_ALPHA=[float(v) for v in _env_list("OSTEO_SWEEP_ALPHA", ["0.1", "0.3", "0.5"])],
    RUN_SWEEP=_env("OSTEO_RUN_SWEEP", 1, int) == 1,
    MAX_EPOCHS=_env("OSTEO_MAX_EPOCHS", 300, int),
    PATIENCE=_env("OSTEO_PATIENCE", 10, int),
    BATCH_SIZE=_env("OSTEO_BATCH", 32, int),
    LR=_env("OSTEO_LR", 1e-4, float),
    STUDENT_DROPOUT=_env("OSTEO_STUDENT_DROPOUT", 0.3, float),
    IMG_SIZE=_env("OSTEO_IMG_SIZE", 224, int),
    SKIP_EXISTING=_env("OSTEO_SKIP_EXISTING", 1, int) == 1,
    SAVE_WEIGHTS=_env("OSTEO_SAVE_WEIGHTS", 1, int) == 1,
    SEED=_env("OSTEO_SEED", 42, int),
    FIT_VERBOSE=_env("OSTEO_FIT_VERBOSE", 0, int),
    LOG_EVERY=_env("OSTEO_LOG_EVERY", 10, int),
    MONITOR="val_loss",
    SELECT_BY=_env("OSTEO_SELECT_BY", "test_f1_macro"),
    # Every training step runs in a fresh process: TensorFlow does not return host memory between runs.
    WORKER_MODE=_env("OSTEO_WORKER_MODE", 1, int) == 1,
    NOTEBOOK=_env("OSTEO_NOTEBOOK", "KD_MultiTeacher_Distillation.ipynb"),
)

TEACHER_KEYS = ["ENB5", "IRV2", "R50"]
STUDENT_ORDER = ["MobileNetV2", "EfficientNetB0", "EfficientNetB2"]
SELECTED_STUDENTS = [s for s in STUDENT_ORDER if s in CFG["STUDENTS"]]
assert SELECTED_STUDENTS, f"No known students in {CFG['STUDENTS']}; choose from {STUDENT_ORDER}"

n_runs = len(SELECTED_STUDENTS) * len(CFG["CONFIGS"])
print(f"Students       : {SELECTED_STUDENTS}")
print(f"Configurations : {CFG['CONFIGS']}")
print(f"Distillations  : {n_runs} runs at T={CFG['TEMPERATURE']:g}, alpha={CFG['ALPHA']:g}"
      + (f" + {len(CFG['SWEEP_T']) * len(CFG['SWEEP_ALPHA'])}-point sweep on the winner" if CFG["RUN_SWEEP"] else ""))
print(f"Protocol       : up to {CFG['MAX_EPOCHS']} epochs, early stopping on {CFG['MONITOR']}, "
      f"patience {CFG['PATIENCE']}, Adam lr {CFG['LR']:g}, batch {CFG['BATCH_SIZE']}")
CFG

### 1.2 Packages

In [ ]:
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys

OPTIONAL_PACKAGES = {
    "numpy": "numpy<2.2", "pandas": "pandas", "cv2": "opencv-python-headless", "PIL": "Pillow", "scipy": "scipy",
    "sklearn": "scikit-learn", "matplotlib": "matplotlib", "seaborn": "seaborn", "scienceplots": "SciencePlots",
    "jinja2": "jinja2",
}
if importlib.util.find_spec("tensorflow") is None:
    raise ImportError('TensorFlow 2.16+ with GPU support is required. On WSL/Linux:\n'
                      '    pip install "tensorflow[and-cuda]==2.19.0"')


def _installed_version(dist):
    try:
        return metadata.version(dist)
    except metadata.PackageNotFoundError:
        return None


missing = [req for module, req in OPTIONAL_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    pins = [f"{dist}=={v}" for dist in ("numpy", "tensorflow") if (v := _installed_version(dist))]
    print("Installing    :", ", ".join(missing))
    print("Kept unchanged:", ", ".join(pins))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing, *pins])
    importlib.invalidate_caches()
    print("Done.")
else:
    print("All required packages are installed.")

### 1.3 Imports

In [ ]:
import gc
import json
import logging
import platform
import shutil
import time
import warnings
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import PIL
from IPython.display import display

import tensorflow as tf

for _gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except RuntimeError as err:
        print("Memory growth not set:", err)

import keras
from tensorflow.keras.applications import (EfficientNetB0, EfficientNetB2, EfficientNetB5, InceptionResNetV2,
                                           MobileNetV2, ResNet50,
                                           efficientnet, inception_resnet_v2, mobilenet_v2, resnet50)
from tensorflow.keras.callbacks import Callback, EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import SGD, Adam, RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import (accuracy_score, auc, classification_report, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.preprocessing import label_binarize

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

warnings.filterwarnings("ignore", message=".*PyDataset.*")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

### 1.4 Versions & GPU check

In [ ]:
assert keras.__version__.startswith("3"), f"Expected Keras 3, found {keras.__version__}"
VERSIONS = {"python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
            "numpy": np.__version__, "opencv": cv2.__version__, "pillow": PIL.__version__}
display(pd.Series(VERSIONS, name="version").to_frame())

TF_GPUS = tf.config.list_physical_devices("GPU")
with tf.device("/GPU:0" if TF_GPUS else "/CPU:0"):
    tf.reduce_sum(tf.matmul(tf.random.normal((256, 256)), tf.random.normal((256, 256)))).numpy()
print("TensorFlow GPU:", [g.name for g in TF_GPUS] or "none - running on CPU")
if not TF_GPUS:
    print("WARNING: no GPU visible. Distillation with three teachers is not practical on CPU.")

### 1.5 Paths

In [ ]:
DATA_DIR = Path(CFG["DATA_DIR"])
TRAIN_DIR, VAL_DIR, TEST_DIR = (DATA_DIR / split for split in ("train", "val", "test"))
for folder in (TRAIN_DIR, VAL_DIR, TEST_DIR):
    assert folder.is_dir(), f"Missing data folder: {folder}"

OUT_DIR = Path(CFG["OUTPUT_DIR"])
TEACHER_DIR = OUT_DIR / "teachers"
SUMMARY_DIR = OUT_DIR / "_summary"
TUNING_DIR = Path(CFG["TUNING_DIR"])
for d in (TEACHER_DIR, SUMMARY_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Data    :", DATA_DIR.resolve())
print("Results :", OUT_DIR.resolve())
print("Phase 2 :", TUNING_DIR.resolve(), "(exists)" if TUNING_DIR.exists() else "(not found - teachers will be retrained)")

## 2. Figure Style

In [ ]:
%matplotlib inline
%config InlineBackend.figure_formats = ["png"]

try:
    import scienceplots  # noqa: F401
    plt.style.use(["science", "ieee", "no-latex"])
    SCIENCEPLOTS = True
except Exception:
    SCIENCEPLOTS = False

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 600, "savefig.bbox": "tight", "savefig.facecolor": "white",
    "font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"], "mathtext.fontset": "stix",
    "font.size": 16, "font.weight": "bold",
    "axes.labelsize": 18, "axes.labelweight": "bold", "axes.titlesize": 18, "axes.titleweight": "bold",
    "xtick.labelsize": 14, "ytick.labelsize": 14,
    "legend.fontsize": 13, "legend.title_fontsize": 14, "figure.titlesize": 20,
    "axes.linewidth": 1.5, "grid.linewidth": 1.0, "lines.linewidth": 2.4, "patch.linewidth": 1.2,
    "xtick.major.width": 1.5, "ytick.major.width": 1.5, "xtick.major.size": 6, "ytick.major.size": 6,
    "legend.frameon": True, "legend.framealpha": 0.95, "legend.edgecolor": "black", "legend.fancybox": True,
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#F0E442", "#000000"]
C_TRAIN, C_VAL, C_REF, C_CHANCE = "#0072B2", "#D55E00", "#222222", "#7F7F7F"
CLASS_COLORS = OKABE_ITO[:5]
STUDENT_COLORS = {"MobileNetV2": "#E69F00", "EfficientNetB0": "#0072B2", "EfficientNetB2": "#009E73"}


def save_fig(fig, stem, save=True):
    if save:
        stem = Path(stem)
        stem.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(f"{stem}.pdf")
        fig.savefig(f"{stem}.png", dpi=600)
    plt.show()
    plt.close(fig)


print("SciencePlots style:", "on" if SCIENCEPLOTS else "not available - using rcParams only")

## 3. Data
Counts per split and grade. Generators yield **raw augmented images** (0-255): each network applies its own `preprocess_input` inside the distiller, because the three teachers and the student use different normalisations.

In [ ]:
CLASS_NAMES = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
assert CLASS_NAMES == ["0", "1", "2", "3", "4"], f"Unexpected class folders: {CLASS_NAMES}"
NUM_CLASSES = len(CLASS_NAMES)
SPLIT_DIRS = {"train": TRAIN_DIR, "val": VAL_DIR, "test": TEST_DIR}

counts = pd.DataFrame({split: {c: sum(f.is_file() for f in (d / c).iterdir()) for c in CLASS_NAMES}
                       for split, d in SPLIT_DIRS.items()})
counts.index.name = "KL grade"
counts.loc["total"] = counts.sum()
display(counts)
N_TRAIN, N_VAL, N_TEST = (int(counts.loc["total", s]) for s in ("train", "val", "test"))

AUGMENTATION = dict(rotation_range=30, width_shift_range=0.2, height_shift_range=0.2, shear_range=0.2,
                    zoom_range=0.2, horizontal_flip=True, brightness_range=[0.8, 1.2], fill_mode="nearest")


def assert_class_order(*generators):
    expected = {c: i for i, c in enumerate(CLASS_NAMES)}
    for g in generators:
        assert g.class_indices == expected, f"Class order mismatch: {g.class_indices}"


def make_raw_generators(batch_size=None, seed=None):
    """Train (augmented, shuffled) and validation generators of RAW images; no preprocess_input."""
    size, bs = (CFG["IMG_SIZE"],) * 2, batch_size or CFG["BATCH_SIZE"]
    train_gen = ImageDataGenerator(**AUGMENTATION).flow_from_directory(
        str(TRAIN_DIR), target_size=size, batch_size=bs, class_mode="categorical", seed=seed)
    val_gen = ImageDataGenerator().flow_from_directory(
        str(VAL_DIR), target_size=size, batch_size=bs, class_mode="categorical", seed=seed)
    assert_class_order(train_gen, val_gen)
    return train_gen, val_gen


def make_raw_eval_generator(folder, batch_size=None):
    """Unshuffled, un-augmented raw generator for scoring."""
    gen = ImageDataGenerator().flow_from_directory(
        str(folder), target_size=(CFG["IMG_SIZE"],) * 2, batch_size=batch_size or CFG["BATCH_SIZE"],
        class_mode="categorical", shuffle=False)
    assert_class_order(gen)
    return gen

## 4. Shared Helpers
Metrics, figures and result files, identical to the earlier notebooks so every phase is comparable.

In [ ]:
class ProgressEvery(Callback):
    """One compact line every `every` epochs instead of one per epoch."""

    def __init__(self, every=10):
        super().__init__()
        self.every = max(1, every)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        parts = " ".join(f"{k} {v:.4f}" for k, v in logs.items() if not k.startswith("val_"))
        val = " ".join(f"{k} {v:.4f}" for k, v in logs.items() if k.startswith("val_"))
        if epoch == 0 or (epoch + 1) % self.every == 0:
            print(f"      epoch {epoch + 1:>3}: {parts} | {val}", flush=True)


class EpochTimer(Callback):
    def on_train_begin(self, logs=None):
        self.epoch_times = []

    def on_epoch_begin(self, epoch, logs=None):
        self._start = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.epoch_times.append(time.perf_counter() - self._start)


def history_info(hist, best_idx, fit_time_s, stopped_early):
    times = hist["epoch_time_s"].to_numpy()
    return dict(
        epochs_run=len(hist), best_epoch=best_idx + 1, stopped_early=bool(stopped_early),
        best_val_loss=float(hist["val_loss"].iloc[best_idx]),
        val_accuracy_at_best=float(hist["val_accuracy"].iloc[best_idx]),
        final_val_accuracy=float(hist["val_accuracy"].iloc[-1]),
        max_val_accuracy=float(hist["val_accuracy"].max()),
        fit_time_s=float(fit_time_s), fit_time_min=float(fit_time_s) / 60,
        time_to_best_s=float(times[: best_idx + 1].sum()),
        epoch1_s=float(times[0]),
        median_epoch_s=float(np.median(times[1:])) if len(times) > 1 else float(times[0]),
    )


def compute_metrics(y_true, prob, prefix="test"):
    labels = list(range(NUM_CLASSES))
    y_pred = prob.argmax(axis=1)
    y_bin = label_binarize(y_true, classes=labels)
    auc_per_class = {}
    for i, c in enumerate(CLASS_NAMES):
        if y_bin[:, i].min() == y_bin[:, i].max():
            auc_per_class[c] = float("nan")
        else:
            fpr, tpr, _ = roc_curve(y_bin[:, i], prob[:, i])
            auc_per_class[c] = float(auc(fpr, tpr))
    fpr_micro, tpr_micro, _ = roc_curve(y_bin.ravel(), prob.ravel())
    kw = dict(zero_division=0)
    scalars = {
        f"{prefix}_accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}_precision_weighted": precision_score(y_true, y_pred, average="weighted", **kw),
        f"{prefix}_recall_weighted": recall_score(y_true, y_pred, average="weighted", **kw),
        f"{prefix}_f1_weighted": f1_score(y_true, y_pred, average="weighted", **kw),
        f"{prefix}_precision_macro": precision_score(y_true, y_pred, average="macro", **kw),
        f"{prefix}_recall_macro": recall_score(y_true, y_pred, average="macro", **kw),
        f"{prefix}_f1_macro": f1_score(y_true, y_pred, average="macro", **kw),
        f"{prefix}_auc_macro": float(np.nanmean(list(auc_per_class.values()))),
        f"{prefix}_auc_micro": auc(fpr_micro, tpr_micro),
    }
    try:
        scalars[f"{prefix}_auc_weighted"] = roc_auc_score(y_bin, prob, average="weighted")
    except ValueError:
        scalars[f"{prefix}_auc_weighted"] = float("nan")
    return dict(
        scalars={k: float(v) for k, v in scalars.items()},
        auc_per_class=auc_per_class,
        confusion_matrix=confusion_matrix(y_true, y_pred, labels=labels),
        report_text=classification_report(y_true, y_pred, labels=labels, target_names=CLASS_NAMES, digits=4, **kw),
        report_dict=classification_report(y_true, y_pred, labels=labels, target_names=CLASS_NAMES,
                                          output_dict=True, **kw),
    )


def plot_curves(name, hist, best_epoch, stem, save=True):
    epochs = np.arange(1, len(hist) + 1)
    marker = "o" if len(hist) <= 30 else None
    fig, axes = plt.subplots(1, 2, figsize=(14.8, 5.6))
    for ax, key, label in ((axes[0], "accuracy", "Accuracy"), (axes[1], "loss", "Loss")):
        ax.plot(epochs, hist[key], color=C_TRAIN, ls="-", lw=2.6, marker=marker, ms=5, label="Training")
        ax.plot(epochs, hist[f"val_{key}"], color=C_VAL, ls="-", lw=2.6, marker=marker, ms=5, label="Validation")
        ax.axvline(best_epoch, color=C_REF, ls=":", lw=2.2, label=f"Best epoch ({best_epoch})")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_xlabel("Epoch")
        ax.set_ylabel(label)
        ax.set_title(label + ("  (train: total KD loss, val: hard-label CE)" if key == "loss" else ""))
        ax.grid(True, alpha=0.3, linestyle="--")
        ax.set_axisbelow(True)
        ax.legend(loc="best")
    fig.suptitle(f"{name}: training and validation", fontweight="bold", y=1.04)
    save_fig(fig, stem, save)


def plot_confusion(name, cm, stem, save=True):
    cm = np.asarray(cm)
    pct = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None) * 100
    annot = np.array([[f"{n}\n({p:.1f}%)" for n, p in zip(row, prow)] for row, prow in zip(cm, pct)])
    fig, ax = plt.subplots(figsize=(8.2, 7.0))
    sns.heatmap(cm, annot=annot, fmt="", cmap="Blues", square=True, linewidths=0.8, linecolor="white",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, annot_kws={"size": 13, "weight": "bold"},
                cbar_kws={"label": "Images"}, ax=ax)
    ax.minorticks_off()
    ax.tick_params(top=False, right=False)
    ax.set_xlabel("Predicted KL grade")
    ax.set_ylabel("True KL grade")
    ax.set_title(f"{name}: confusion matrix (test)")
    save_fig(fig, stem, save)


def plot_roc(name, y_true, prob, res, stem, save=True):
    y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(7.6, 7.0))
    for i, c in enumerate(CLASS_NAMES):
        if y_bin[:, i].min() == y_bin[:, i].max():
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], prob[:, i])
        ax.plot(fpr, tpr, color=CLASS_COLORS[i], ls="-", lw=2.4, label=f"KL {c} (AUC = {res['auc_per_class'][c]:.3f})")
    fpr, tpr, _ = roc_curve(y_bin.ravel(), prob.ravel())
    ax.plot(fpr, tpr, color=C_REF, ls=":", lw=2.6, label=f"Micro-average (AUC = {res['test_auc_micro']:.3f})")
    ax.plot([0, 1], [0, 1], color=C_CHANCE, ls="--", lw=1.8, label="Chance")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.set_aspect("equal")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(f"{name}: ROC (test)\nmacro AUC = {res['test_auc_macro']:.3f}")
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)
    ax.legend(loc="lower right", fontsize=11)
    save_fig(fig, stem, save)


def _json_default(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Not JSON serialisable: {type(obj)}")


def write_json_atomic(path, data):
    tmp = Path(f"{path}.tmp")
    tmp.write_text(json.dumps(data, indent=2, default=_json_default))
    os.replace(tmp, path)


def print_result(name, res, report_path=None):
    print(f"\n{'=' * 70}\n{name}: test results\n{'=' * 70}")
    print(f"Test accuracy      : {res['test_accuracy']:.4f}")
    print(f"F1 macro / weighted: {res['test_f1_macro']:.4f} / {res['test_f1_weighted']:.4f}")
    print(f"Precision / recall : {res['test_precision_macro']:.4f} / {res['test_recall_macro']:.4f} (macro)")
    print(f"ROC AUC macro      : {res['test_auc_macro']:.4f}")
    if "params_total" in res:
        print(f"Parameters         : {res['params_total']:,}   ({res.get('size_mb', float('nan')):.1f} MB)")
    if "ms_per_image" in res:
        print(f"Inference          : {res['ms_per_image']:.2f} ms/image (batch {CFG['BATCH_SIZE']})")
    if "fit_time_min" in res:
        print(f"Training time      : {res['fit_time_min']:.1f} min over {res.get('epochs_run', '?')} epochs")
    if report_path and Path(report_path).exists():
        print("\nClassification report (test):\n" + Path(report_path).read_text())


def finalize_run(name, out, info, y_true, prob, filenames, hist=None):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    assert np.allclose(prob.sum(axis=1), 1.0, atol=1e-3), "Class probabilities do not sum to 1"
    m = compute_metrics(y_true, prob, "test")

    if hist is not None:
        hist.to_csv(out / "history.csv", index=False)
    pred = pd.DataFrame({"file": filenames, "y_true": y_true, "y_pred": prob.argmax(axis=1)})
    for i, c in enumerate(CLASS_NAMES):
        pred[f"prob_{c}"] = prob[:, i]
    pred.to_csv(out / "test_predictions.csv", index=False)
    (out / "classification_report.txt").write_text(m["report_text"])
    pd.DataFrame(m["report_dict"]).T.to_csv(out / "classification_report.csv")
    pd.DataFrame(m["confusion_matrix"], index=[f"true_{c}" for c in CLASS_NAMES],
                 columns=[f"pred_{c}" for c in CLASS_NAMES]).to_csv(out / "confusion_matrix.csv")

    res = {**info, **m["scalars"], "auc_per_class": m["auc_per_class"],
           "confusion_matrix": m["confusion_matrix"].tolist(),
           "finished_at": datetime.now().isoformat(timespec="seconds")}

    fig_dir = out / "figures"
    if hist is not None:
        plot_curves(name, hist, info.get("best_epoch", len(hist)), fig_dir / "training_curves")
    plot_confusion(name, m["confusion_matrix"], fig_dir / "confusion_matrix")
    plot_roc(name, y_true, prob, {**res, **m}, fig_dir / "roc_curve")

    write_json_atomic(out / "metrics.json", res)
    print_result(name, res, out / "classification_report.txt")
    return res


def load_result(name, out, show=True):
    out = Path(out)
    res = json.loads((out / "metrics.json").read_text())
    if show:
        pred = pd.read_csv(out / "test_predictions.csv")
        prob = pred[[f"prob_{c}" for c in CLASS_NAMES]].to_numpy()
        y_true = pred["y_true"].to_numpy()
        m = compute_metrics(y_true, prob, "test")
        if (out / "history.csv").exists():
            hist = pd.read_csv(out / "history.csv")
            plot_curves(name, hist, res.get("best_epoch", len(hist)), out / "figures" / "training_curves", save=False)
        plot_confusion(name, m["confusion_matrix"], out / "figures" / "confusion_matrix", save=False)
        plot_roc(name, y_true, prob, {**res, **m}, out / "figures" / "roc_curve", save=False)
        print_result(name, res, out / "classification_report.txt")
    return res


def cleanup_backend():
    keras.backend.clear_session()
    gc.collect()


def measure_latency(model, preprocess_fn, n_batches=5):
    """Milliseconds per image at batch CFG['BATCH_SIZE'], after a warm-up batch."""
    x = np.random.rand(CFG["BATCH_SIZE"], CFG["IMG_SIZE"], CFG["IMG_SIZE"], 3).astype("float32") * 255
    xp = preprocess_fn(x.copy())
    model(xp, training=False)
    start = time.perf_counter()
    for _ in range(n_batches):
        model(xp, training=False)
    elapsed = time.perf_counter() - start
    return float(elapsed / (n_batches * CFG["BATCH_SIZE"]) * 1000)


RESULTS = {}

## 5. Teachers
Each teacher is loaded from phase 2 if its weights are there, otherwise retrained from its recorded winning configuration. The final softmax is then replaced by a **logits** output (the fusion happens before any softmax), and each teacher is evaluated on the test set so the numbers reported here belong to the weights actually used.

In [ ]:
TEACHERS = {
    "ENB5": dict(model="Custom-ENB5", ctor=EfficientNetB5, pre=efficientnet.preprocess_input,
                 source=TUNING_DIR / "Custom-ENB5" / "best.keras",
                 config_json=TUNING_DIR / "Custom-ENB5" / "metrics.json",
                 fallback=dict(learning_rate=1e-4, dropout_rate=0.3, optimizer="rmsprop", batch_size=32)),
    "IRV2": dict(model="InceptionResNetV2", ctor=InceptionResNetV2, pre=inception_resnet_v2.preprocess_input,
                 source=TUNING_DIR / "InceptionResNetV2" / "best" / "best.keras",
                 config_json=TUNING_DIR / "InceptionResNetV2" / "best" / "combo.json",
                 fallback=dict(learning_rate=1e-4, dropout_rate=0.5, optimizer="adam", batch_size=16)),
    "R50": dict(model="ResNet50", ctor=ResNet50, pre=resnet50.preprocess_input,
                source=TUNING_DIR / "ResNet50" / "best" / "best.keras",
                config_json=TUNING_DIR / "ResNet50" / "best" / "combo.json",
                fallback=dict(learning_rate=1e-4, dropout_rate=0.3, optimizer="adam", batch_size=32)),
}
assert list(TEACHERS) == TEACHER_KEYS


def teacher_config(key):
    """The winning hyperparameters from phase 2, or the recorded fallback."""
    spec = TEACHERS[key]
    if Path(spec["config_json"]).exists():
        r = json.loads(Path(spec["config_json"]).read_text())
        if all(k in r for k in ("learning_rate", "dropout_rate", "optimizer", "batch_size")):
            return {k: r[k] for k in ("learning_rate", "dropout_rate", "optimizer", "batch_size")}
    return dict(spec["fallback"])


def build_teacher_model(ctor, dropout_rate):
    """Phase-2 recipe: unfrozen backbone + GAP + 1024/512/1024/512 swish head + softmax."""
    size = CFG["IMG_SIZE"]
    base = ctor(weights="imagenet", include_top=False, input_shape=(size, size, 3))
    x = GlobalAveragePooling2D()(base.output)
    for units in (1024, 512, 1024, 512):
        x = Dense(units, activation="swish")(x)
        x = Dropout(dropout_rate)(x)
    return Model(base.input, Dense(NUM_CLASSES, activation="softmax")(x))


def make_optimizer(name, learning_rate):
    return {"rmsprop": RMSprop, "adam": Adam}[name](learning_rate=learning_rate) if name != "sgd" \
        else SGD(learning_rate=learning_rate, momentum=0.9)


def train_teacher(key):
    """Retrain one teacher at its phase-2 winning configuration (only if its weights are missing)."""
    spec, cfg = TEACHERS[key], teacher_config(key)
    print(f"{spec['model']}: retraining at {cfg}")
    keras.utils.set_random_seed(CFG["SEED"])
    train_gen, val_gen = make_raw_generators(batch_size=int(cfg["batch_size"]), seed=CFG["SEED"])
    pre = spec["pre"]
    model = build_teacher_model(spec["ctor"], float(cfg["dropout_rate"]))
    model.compile(optimizer=make_optimizer(cfg["optimizer"], float(cfg["learning_rate"])),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    early = EarlyStopping(monitor="val_loss", patience=CFG["PATIENCE"], restore_best_weights=True, verbose=1)

    class Preprocessed(keras.utils.PyDataset):        # the generators are raw; apply this teacher's preprocessing
        def __init__(self, gen):
            super().__init__()
            self.gen = gen

        def __len__(self):
            return len(self.gen)

        def __getitem__(self, i):
            x, y = self.gen[i]
            return pre(x.copy()), y

        def on_epoch_end(self):
            self.gen.on_epoch_end()

    model.fit(Preprocessed(train_gen), epochs=CFG["MAX_EPOCHS"], validation_data=Preprocessed(val_gen),
              callbacks=[early, ProgressEvery(CFG["LOG_EVERY"])], verbose=CFG["FIT_VERBOSE"])
    if early.best_weights is not None:
        model.set_weights(early.best_weights)
    return model


def to_logits_model(model):
    """Replace the final softmax Dense by an identical linear one: fusion happens on logits."""
    dense = model.layers[-1]
    assert isinstance(dense, Dense) and dense.units == NUM_CLASSES, f"Unexpected final layer: {dense}"
    logits = Dense(NUM_CLASSES, activation=None, name="logits")(dense.input)
    logit_model = Model(model.input, logits)
    logit_model.get_layer("logits").set_weights(dense.get_weights())
    return logit_model


def prepare_teacher(key):
    """Ensure results/teachers/<Teacher>/teacher.keras exists and is evaluated; return its metrics."""
    spec = TEACHERS[key]
    out = TEACHER_DIR / spec["model"]
    out.mkdir(parents=True, exist_ok=True)
    target = out / "teacher.keras"

    if not target.exists():
        if Path(spec["source"]).exists():
            print(f"{spec['model']}: copying weights from {spec['source']}")
            shutil.copy2(spec["source"], target)
        else:
            model = train_teacher(key)
            model.save(target)
            del model
            cleanup_backend()

    if CFG["SKIP_EXISTING"] and (out / "metrics.json").exists():
        return load_result(spec["model"], out, show=False)

    model = keras.models.load_model(target)
    logit_model = to_logits_model(model)
    test_gen = make_raw_eval_generator(TEST_DIR)
    probs, logits_all = [], []
    for i in range(len(test_gen)):
        x, _ = test_gen[i]
        z = logit_model(spec["pre"](x.copy()), training=False).numpy()
        logits_all.append(z)
        probs.append(tf.nn.softmax(z).numpy())
    prob = np.concatenate(probs)
    np.save(out / "test_logits.npy", np.concatenate(logits_all))    # reused by the ensemble
    # the logits model must reproduce the original softmax outputs
    check = model.predict(spec["pre"](test_gen[0][0].copy()), verbose=0)
    assert np.allclose(check, tf.nn.softmax(logits_all[0]).numpy(), atol=1e-4), "logit conversion changed the outputs"

    info = dict(model=spec["model"], role="teacher", key=key, img_size=CFG["IMG_SIZE"],
                params_total=int(model.count_params()), size_mb=float(target.stat().st_size / 2**20),
                ms_per_image=measure_latency(model, spec["pre"]), versions=VERSIONS, **teacher_config(key))
    res = finalize_run(spec["model"], out, info, test_gen.classes, prob, test_gen.filenames)
    del model, logit_model
    cleanup_backend()
    return res

### 5.1 Prepare and evaluate the three teachers

In [ ]:
for key in TEACHER_KEYS:                                                            # worker: skip
    RESULTS[TEACHERS[key]["model"]] = prepare_teacher(key)

### 5.2 The fused teacher ensemble
The equal-weight logit fusion evaluated directly on the test set: this is the ceiling the students are distilled towards.

In [ ]:
# worker: skip
ens_dir = TEACHER_DIR / "_ensemble"
if CFG["SKIP_EXISTING"] and (ens_dir / "metrics.json").exists():
    RESULTS["Fused-teachers"] = load_result("Fused teachers", ens_dir)
else:
    logits = [np.load(TEACHER_DIR / TEACHERS[k]["model"] / "test_logits.npy") for k in TEACHER_KEYS]
    fused = np.mean(logits, axis=0)                      # equal weights, w_i = 1/3
    prob = tf.nn.softmax(fused).numpy()
    test_gen = make_raw_eval_generator(TEST_DIR)
    info = dict(model="Fused teachers (ENB5+IRV2+R50)", role="ensemble", fusion="mean of logits",
                params_total=int(sum(RESULTS[TEACHERS[k]["model"]]["params_total"] for k in TEACHER_KEYS)),
                size_mb=float(sum(RESULTS[TEACHERS[k]["model"]]["size_mb"] for k in TEACHER_KEYS)),
                ms_per_image=float(sum(RESULTS[TEACHERS[k]["model"]]["ms_per_image"] for k in TEACHER_KEYS)),
                versions=VERSIONS)
    RESULTS["Fused-teachers"] = finalize_run("Fused teachers", ens_dir, info, test_gen.classes, prob,
                                             test_gen.filenames)

## 6. The Distiller
`MultiTeacherDistiller` implements the diagram: teachers run in inference mode on their own preprocessing, their logits are averaged, temperature-scaled into soft targets, and combined with the hard-label cross-entropy.

- **Training loss** = $\alpha L_{CE} + (1-\alpha)T^2\mathrm{KL}(P_T\parallel P_S)$
- **Validation loss** = hard-label cross-entropy only, so early stopping and model selection mean the same thing for every configuration, including the no-teacher control.

In [ ]:
STUDENTS = {
    "MobileNetV2": dict(ctor=MobileNetV2, pre=mobilenet_v2.preprocess_input),
    "EfficientNetB0": dict(ctor=EfficientNetB0, pre=efficientnet.preprocess_input),
    "EfficientNetB2": dict(ctor=EfficientNetB2, pre=efficientnet.preprocess_input),
}
assert list(STUDENTS) == STUDENT_ORDER


def build_student(name, dropout_rate=None):
    """Compact KD student: unfrozen backbone + GAP + Dense(256, swish) + Dropout + Dense(5) logits."""
    size = CFG["IMG_SIZE"]
    base = STUDENTS[name]["ctor"](weights="imagenet", include_top=False, input_shape=(size, size, 3))
    x = GlobalAveragePooling2D()(base.output)
    x = Dense(256, activation="swish")(x)
    x = Dropout(CFG["STUDENT_DROPOUT"] if dropout_rate is None else dropout_rate)(x)
    logits = Dense(NUM_CLASSES, activation=None, name="logits")(x)
    return Model(base.input, logits)


class _TeacherBundle:
    """Holds the teachers outside Keras's tracking, so early stopping and save() touch only the student."""

    def __init__(self, models, preprocessors):
        self.models = list(models)
        self.pres = list(preprocessors)
        for m in self.models:
            m.trainable = False

    def __len__(self):
        return len(self.models)

    def logits(self, x):
        return [m(pre(x), training=False) for m, pre in zip(self.models, self.pres)]


class MultiTeacherDistiller(keras.Model):
    """Fuses teacher logits, distils into the student, and adds hard-label supervision."""

    def __init__(self, student, student_pre, teachers=(), teacher_pres=(), weights=None,
                 temperature=3.0, alpha=0.3):
        super().__init__()
        self.student = student
        self.student_pre = student_pre
        self._bundle = _TeacherBundle(teachers, teacher_pres)   # not tracked: see _TeacherBundle
        n = len(self._bundle)
        self.n_teachers = n
        self.w = [1.0 / n] * n if (weights is None and n) else list(weights or [])
        self.T = float(temperature)
        self.alpha = float(alpha) if n else 1.0          # no teacher -> pure cross-entropy
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.ce_tracker = keras.metrics.Mean(name="ce")
        self.kd_tracker = keras.metrics.Mean(name="kd")
        self.acc_tracker = keras.metrics.CategoricalAccuracy(name="accuracy")
        self.val_loss_tracker = keras.metrics.Mean(name="loss")
        self.val_acc_tracker = keras.metrics.CategoricalAccuracy(name="accuracy")

    @property
    def metrics(self):
        return [self.loss_tracker, self.ce_tracker, self.kd_tracker, self.acc_tracker]

    def call(self, x, training=False):
        return self.student(self.student_pre(x), training=training)

    def _compute_losses(self, x, y, training):
        student_logits = self.student(self.student_pre(x), training=training)
        ce = tf.reduce_mean(keras.losses.categorical_crossentropy(y, student_logits, from_logits=True))
        kd = tf.constant(0.0)
        if self.n_teachers:
            teacher_logits = self._bundle.logits(x)
            fused = tf.add_n([w * z for w, z in zip(self.w, teacher_logits)])          # zF = sum wi zi
            p_t = tf.nn.softmax(fused / self.T)                                        # PT
            log_p_s = tf.nn.log_softmax(student_logits / self.T)                       # log PS
            log_p_t = tf.nn.log_softmax(fused / self.T)
            kd = tf.reduce_mean(tf.reduce_sum(p_t * (log_p_t - log_p_s), axis=1)) * (self.T ** 2)
        return student_logits, ce, kd, self.alpha * ce + (1.0 - self.alpha) * kd

    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            logits, ce, kd, loss = self._compute_losses(x, y, training=True)
        grads = tape.gradient(loss, self.student.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.student.trainable_variables))
        self.loss_tracker.update_state(loss)
        self.ce_tracker.update_state(ce)
        self.kd_tracker.update_state(kd)
        self.acc_tracker.update_state(y, logits)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x, y = data
        logits = self.student(self.student_pre(x), training=False)
        ce = tf.reduce_mean(keras.losses.categorical_crossentropy(y, logits, from_logits=True))
        self.val_loss_tracker.update_state(ce)           # validation loss = hard-label CE
        self.val_acc_tracker.update_state(y, logits)
        return {"loss": self.val_loss_tracker.result(), "accuracy": self.val_acc_tracker.result()}


def load_teacher_logit_models(keys):
    models, pres = [], []
    for k in keys:
        spec = TEACHERS[k]
        model = keras.models.load_model(TEACHER_DIR / spec["model"] / "teacher.keras")
        models.append(to_logits_model(model))
        pres.append(spec["pre"])
    return models, pres


def run_config(student_name, config, temperature=None, alpha=None, out=None):
    """Train one student under one teacher configuration, then evaluate it on the test set."""
    T = CFG["TEMPERATURE"] if temperature is None else float(temperature)
    alpha = CFG["ALPHA"] if alpha is None else float(alpha)
    keys = [] if config == "none" else config.split("+")
    assert all(k in TEACHERS for k in keys), f"Unknown teacher in {config}"
    if not keys:
        alpha = 1.0     # the control has no teacher: pure hard-label cross-entropy, recorded as such
    out = Path(out) if out else OUT_DIR / student_name / config
    out.mkdir(parents=True, exist_ok=True)

    keras.utils.set_random_seed(CFG["SEED"])
    train_gen, val_gen = make_raw_generators(seed=CFG["SEED"])
    student = build_student(student_name)
    teachers, teacher_pres = load_teacher_logit_models(keys)
    distiller = MultiTeacherDistiller(student, STUDENTS[student_name]["pre"], teachers, teacher_pres,
                                      temperature=T, alpha=alpha)
    distiller.compile(optimizer=Adam(learning_rate=CFG["LR"]))

    early = EarlyStopping(monitor="val_loss", patience=CFG["PATIENCE"], restore_best_weights=True, verbose=1)
    timer = EpochTimer()
    start = time.perf_counter()
    history = distiller.fit(train_gen, epochs=CFG["MAX_EPOCHS"], validation_data=val_gen,
                            callbacks=[early, timer, ProgressEvery(CFG["LOG_EVERY"])], verbose=CFG["FIT_VERBOSE"])
    fit_time = time.perf_counter() - start

    hist = pd.DataFrame(history.history)
    hist.insert(0, "epoch", np.arange(1, len(hist) + 1))
    hist["epoch_time_s"] = timer.epoch_times
    best_idx = int(getattr(early, "best_epoch", hist["val_loss"].to_numpy().argmin()))
    if early.best_weights is not None:
        distiller.set_weights(early.best_weights)

    test_gen = make_raw_eval_generator(TEST_DIR)
    probs = []
    for i in range(len(test_gen)):
        x, _ = test_gen[i]
        probs.append(tf.nn.softmax(student(STUDENTS[student_name]["pre"](x.copy()), training=False)).numpy())
    prob = np.concatenate(probs)

    if CFG["SAVE_WEIGHTS"]:
        student.save(out / "student.keras")
    info = dict(model=f"{student_name} [{config}]", student=student_name, config=config, role="student",
                teachers=keys, n_teachers=len(keys), temperature=T, alpha=alpha, fusion="mean of logits",
                img_size=CFG["IMG_SIZE"], batch_size=CFG["BATCH_SIZE"], learning_rate=CFG["LR"],
                params_total=int(student.count_params()),
                size_mb=float((out / "student.keras").stat().st_size / 2**20) if CFG["SAVE_WEIGHTS"] else float("nan"),
                ms_per_image=measure_latency(student, STUDENTS[student_name]["pre"]),
                max_epochs=CFG["MAX_EPOCHS"], patience=CFG["PATIENCE"], seed=CFG["SEED"], versions=VERSIONS,
                n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST)
    info.update(history_info(hist, best_idx, fit_time, early.stopped_epoch > 0))
    res = finalize_run(f"{student_name} [{config}]", out, info, test_gen.classes, prob, test_gen.filenames, hist=hist)
    del distiller, student, teachers
    cleanup_backend()

    return res

### 6.1 Worker
Each distillation runs in a fresh process, for the same reason as phase 2: TensorFlow does not return host memory between runs, and three teachers plus a student is a lot of memory to leak 24 times over.

In [ ]:
WORKER_PATH = Path("kd_worker.py")
WORKER_SOURCE = r"""
# Run ONE distillation in a fresh process, then exit.
# Written by the KD notebook; safe to delete. It executes the notebook's own setup cells, so the notebook
# stays the single source of truth for the recipe.
#   run NOTEBOOK STUDENT CONFIG T ALPHA OUTDIR
import sys
from pathlib import Path

import nbformat

MODE, NOTEBOOK = sys.argv[1], sys.argv[2]
SKIP_TAG = '# worker' + ': skip'   # assembled, so this line cannot match itself

ns = {}
for cell in nbformat.read(NOTEBOOK, as_version=4).cells:
    if cell.cell_type != 'code':
        continue
    src = chr(10).join(l for l in cell.source.splitlines() if not l.lstrip().startswith(('%', '!')))
    if SKIP_TAG in src:
        continue
    exec(compile(src, '<notebook>', 'exec'), ns)

for needed in ('run_config', 'prepare_teacher', 'OUT_DIR'):
    assert needed in ns, 'the worker skipped the notebook cell defining ' + needed

if MODE == 'run':
    STUDENT, CONFIG = sys.argv[3], sys.argv[4]
    T, ALPHA, OUTDIR = float(sys.argv[5]), float(sys.argv[6]), sys.argv[7]
    ns['run_config'](STUDENT, CONFIG, temperature=T, alpha=ALPHA, out=Path(OUTDIR))
elif MODE == 'teacher':
    ns['prepare_teacher'](sys.argv[3])
else:
    raise SystemExit('unknown worker mode: ' + MODE)
"""


def write_worker():
    assert Path(CFG["NOTEBOOK"]).exists(), (
        f"Cannot find {CFG['NOTEBOOK']} in {Path.cwd()}. Set CFG['NOTEBOOK'], or CFG['WORKER_MODE'] = False.")
    if not WORKER_PATH.exists() or WORKER_PATH.read_text() != WORKER_SOURCE:
        WORKER_PATH.write_text(WORKER_SOURCE)
    return WORKER_PATH


def run_in_worker(mode, *args):
    """Run one step in a fresh process, streaming its output. Returns the exit code."""
    cmd = [sys.executable, str(write_worker()), mode, CFG["NOTEBOOK"], *[str(a) for a in args]]
    env = {**os.environ, "MPLBACKEND": "Agg", "OSTEO_WORKER_MODE": "0"}
    noise = ("Found ", "Students  ", "Configurations", "Distillations", "Protocol", "Data    ", "Results :",
             "Phase 2 :", "SciencePlots", "All required", "TensorFlow GPU")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in proc.stdout:
        line = line.rstrip()
        if line and not line.startswith(noise):
            print("   ", line[:300], flush=True)
    return proc.wait()


def run_or_load(student_name, config, temperature=None, alpha=None, out=None, label=None):
    """Train one configuration (in a worker unless disabled), skipping it if its results already exist."""
    T = CFG["TEMPERATURE"] if temperature is None else float(temperature)
    alpha = CFG["ALPHA"] if alpha is None else float(alpha)
    out = Path(out) if out else OUT_DIR / student_name / config
    label = label or f"{student_name} [{config}]"
    if CFG["SKIP_EXISTING"] and (out / "metrics.json").exists():
        print(f"{label}: finished results found - loading instead of retraining")
        res = load_result(label, out)
    elif CFG["WORKER_MODE"]:
        print(f"\n=== {label}: T={T:g}, alpha={alpha:g}")
        code = run_in_worker("run", student_name, config, T, alpha, str(out))
        if code != 0 or not (out / "metrics.json").exists():
            print(f"{label}: FAILED (worker exited with code {code})")
            return None
        res = load_result(label, out)
    else:
        res = run_config(student_name, config, temperature=T, alpha=alpha, out=out)
    RESULTS[label] = res
    return res

## 7. Distillation Runs
Every student is trained under all 8 configurations at T=3, alpha=0.3. `none` is the control: the same student, same recipe, hard labels only.

### 7.1 MobileNetV2

In [ ]:
if "MobileNetV2" in SELECTED_STUDENTS:  # worker: skip
    for _config in CFG["CONFIGS"]:
        run_or_load("MobileNetV2", _config)
else:
    print("MobileNetV2: not in STUDENTS - skipped")

### 7.2 EfficientNetB0

In [ ]:
if "EfficientNetB0" in SELECTED_STUDENTS:  # worker: skip
    for _config in CFG["CONFIGS"]:
        run_or_load("EfficientNetB0", _config)
else:
    print("EfficientNetB0: not in STUDENTS - skipped")

### 7.3 EfficientNetB2

In [ ]:
if "EfficientNetB2" in SELECTED_STUDENTS:  # worker: skip
    for _config in CFG["CONFIGS"]:
        run_or_load("EfficientNetB2", _config)
else:
    print("EfficientNetB2: not in STUDENTS - skipped")

## 8. Temperature / Alpha Sweep
A 3x3 sweep (T in 2/3/5, alpha in 0.1/0.3/0.5) on the best student+configuration found above, to check the KD hyperparameters were not simply lucky. The T=3, alpha=0.3 point is reused from section 7.

In [ ]:
# worker: skip
def best_run():
    rows = [json.loads(p.read_text()) for p in OUT_DIR.glob("*/*/metrics.json")
            if json.loads(p.read_text()).get("role") == "student"]
    rows = [r for r in rows if r.get("config") != "none"]
    return max(rows, key=lambda r: r[CFG["SELECT_BY"]]) if rows else None


winner = best_run()
if not CFG["RUN_SWEEP"]:
    print("Sweep disabled in CFG - skipped")
elif winner is None:
    print("No distillation runs finished yet - nothing to sweep")
else:
    print(f"Sweeping around the best run: {winner['student']} [{winner['config']}] "
          f"({CFG['SELECT_BY']} = {winner[CFG['SELECT_BY']]:.4f})")
    for T in CFG["SWEEP_T"]:
        for alpha in CFG["SWEEP_ALPHA"]:
            tag = f"T{T:g}_a{alpha:g}"
            out = OUT_DIR / "_sweep" / winner["student"] / winner["config"] / tag
            if T == CFG["TEMPERATURE"] and alpha == CFG["ALPHA"]:
                out = OUT_DIR / winner["student"] / winner["config"]        # already trained in section 7
            run_or_load(winner["student"], winner["config"], temperature=T, alpha=alpha, out=out,
                        label=f"{winner['student']} [{winner['config']}] {tag}")

## 9. Summary
### 9.1 Every model in one table
Teachers, the fused ensemble, and every student configuration, with size, latency and test metrics.

In [ ]:
# worker: skip
def collect(path_glob, role=None):
    rows = []
    for p in sorted(OUT_DIR.glob(path_glob)):
        r = json.loads(p.read_text())
        if role is None or r.get("role") == role:
            rows.append(r)
    return rows


records = collect("teachers/*/metrics.json") + collect("*/*/metrics.json", role="student")
print(f"Models finished: {len(records)}")
COLUMNS = ["model", "role", "student", "config", "n_teachers", "temperature", "alpha", "params_total", "size_mb",
           "ms_per_image", "epochs_run", "best_epoch", "val_accuracy_at_best", "test_accuracy",
           "test_precision_macro", "test_recall_macro", "test_f1_macro", "test_f1_weighted", "test_auc_macro",
           "fit_time_min"]
summary = pd.DataFrame(records).reindex(columns=COLUMNS)
summary["params_M"] = summary["params_total"] / 1e6
summary.to_csv(SUMMARY_DIR / "summary.csv", index=False)

advisor = summary[["model", "params_M", "val_accuracy_at_best", "test_accuracy", "test_f1_macro",
                   "ms_per_image", "fit_time_min"]].rename(columns={
    "model": "Model", "params_M": "Params (M)", "val_accuracy_at_best": "Validation Accuracy",
    "test_accuracy": "Test Accuracy", "test_f1_macro": "Test Macro F1",
    "ms_per_image": "Inference (ms/img)", "fit_time_min": "Training Time (min)"})
advisor.to_csv(SUMMARY_DIR / "advisor_table.csv", index=False)

sweep_rows = collect("_sweep/*/*/*/metrics.json", role="student")
if sweep_rows:
    pd.DataFrame(sweep_rows)[["student", "config", "temperature", "alpha", "test_accuracy", "test_f1_macro",
                              "val_accuracy_at_best", "epochs_run"]].sort_values(
        "test_f1_macro", ascending=False).to_csv(SUMMARY_DIR / "sweep.csv", index=False)

shown = summary.sort_values("test_f1_macro", ascending=False).reset_index(drop=True)
metric_cols = ["test_accuracy", "test_f1_macro", "test_f1_weighted", "test_auc_macro"]
try:
    display(shown.style.format(precision=4, na_rep="-").highlight_max(subset=metric_cols, color="#cfe8ff"))
except Exception:
    display(shown.round(4))
print("Saved:", SUMMARY_DIR / "summary.csv")

### 9.2 Which student is best
Best configuration per student, the gain distillation bought over the no-teacher control, and how close each student gets to its teachers.

In [ ]:
# worker: skip
students = summary[summary.role == "student"].copy()
if students.empty:
    print("No student runs yet.")
else:
    best_per_student = students.sort_values("test_f1_macro", ascending=False).groupby("student").head(1)
    control = students[students.config == "none"].set_index("student")["test_f1_macro"]
    rows = []
    for _, r in best_per_student.iterrows():
        base = control.get(r["student"], float("nan"))
        rows.append({"student": r["student"], "params_M": round(r["params_M"], 2),
                     "best config": r["config"], "test macro F1": round(r["test_f1_macro"], 4),
                     "test accuracy": round(r["test_accuracy"], 4),
                     "control (no teacher)": round(base, 4), "KD gain": round(r["test_f1_macro"] - base, 4),
                     "ms/img": round(r["ms_per_image"], 2)})
    table = pd.DataFrame(rows).sort_values("test macro F1", ascending=False)
    display(table)
    table.to_csv(SUMMARY_DIR / "best_per_student.csv", index=False)

    teachers_df = summary[summary.role.isin(["teacher", "ensemble"])]
    if not teachers_df.empty:
        ceiling = teachers_df["test_f1_macro"].max()
        top = table.iloc[0]
        print(f"\nBest student : {top['student']} [{top['best config']}] - macro F1 {top['test macro F1']:.4f} "
              f"at {top['params_M']:.1f}M params")
        print(f"Best teacher : macro F1 {ceiling:.4f} at {teachers_df['params_M'].max():.1f}M params")
        print(f"The student keeps {100 * top['test macro F1'] / ceiling:.1f}% of the best teacher's macro F1 "
              f"with {teachers_df['params_M'].max() / top['params_M']:.1f}x fewer parameters")

### 9.3 Figures
Performance versus size, the headline test metrics, a teacher/student comparison, training time, and all validation curves.

In [ ]:
# worker: skip
FIG_SUM = SUMMARY_DIR / "figures"

if summary.empty:
    print("Nothing to plot yet.")
else:
    # 1) performance vs size -------------------------------------------------
    fig, ax = plt.subplots(figsize=(9.2, 6.2))
    tch = summary[summary.role == "teacher"]
    ens = summary[summary.role == "ensemble"]
    ax.scatter(tch["params_M"], tch["test_f1_macro"], s=260, marker="s", color=C_REF, edgecolor="black",
               linewidth=1.2, label="Teachers", zorder=3)
    for _, r in tch.iterrows():
        ax.annotate(r["model"], (r["params_M"], r["test_f1_macro"]), textcoords="offset points",
                    xytext=(8, -4), fontsize=10, fontweight="bold")
    if not ens.empty:
        ax.scatter(ens["params_M"], ens["test_f1_macro"], s=300, marker="*", color="#D55E00",
                   edgecolor="black", linewidth=1.2, label="Fused teachers", zorder=3)
    for student_name, grp in summary[summary.role == "student"].groupby("student"):
        grp = grp.sort_values("params_M")
        colour = STUDENT_COLORS.get(student_name, "#7F7F7F")
        ax.scatter(grp["params_M"], grp["test_f1_macro"], s=110, color=colour, edgecolor="black",
                   linewidth=1.0, label=student_name, zorder=3, alpha=0.9)
        best = grp.loc[grp["test_f1_macro"].idxmax()]
        ax.annotate(f"{student_name}\n[{best['config']}]", (best["params_M"], best["test_f1_macro"]),
                    textcoords="offset points", xytext=(6, 8), fontsize=9, fontweight="bold", color=colour)
    ax.set_xscale("log")
    ax.set_xlabel("Parameters (millions, log scale)")
    ax.set_ylabel("Test macro F1")
    ax.set_title("Performance vs model size")
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)
    ax.legend(loc="lower right", fontsize=11)
    save_fig(fig, FIG_SUM / "performance_vs_size")

    # 2) test metrics per model ---------------------------------------------
    d = summary.sort_values(["role", "test_f1_macro"], ascending=[True, False])
    x = np.arange(len(d))
    metrics = [("test_accuracy", "Accuracy"), ("test_f1_macro", "Macro F1"),
               ("test_f1_weighted", "Weighted F1"), ("test_auc_macro", "Macro AUC")]
    w = 0.8 / len(metrics)
    fig, ax = plt.subplots(figsize=(max(9.0, 0.75 * len(d)), 5.8))
    for i, (key, label) in enumerate(metrics):
        ax.bar(x + (i - (len(metrics) - 1) / 2) * w, d[key], w, label=label, color=OKABE_ITO[i],
               edgecolor="black", linewidth=1.0)
    ax.set_xticks(x)
    ax.set_xticklabels(d["model"], rotation=40, ha="right", fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_yticks(np.arange(0, 1.01, 0.2))
    ax.set_ylabel("Score (test set)")
    ax.set_title("Test-set performance: teachers and distilled students")
    ax.grid(True, axis="y", alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)
    ax.legend(ncol=4, fontsize=11, loc="upper center")
    save_fig(fig, FIG_SUM / "test_metrics")

    # 3) configuration comparison per student --------------------------------
    st = summary[summary.role == "student"]
    if not st.empty:
        configs = [c for c in CFG["CONFIGS"] if c in set(st["config"])]
        fig, ax = plt.subplots(figsize=(max(9.0, 1.1 * len(configs)), 5.8))
        names = sorted(st["student"].unique())
        w = 0.8 / max(1, len(names))
        xs = np.arange(len(configs))
        for i, student_name in enumerate(names):
            vals = [st[(st.student == student_name) & (st.config == c)]["test_f1_macro"].max() for c in configs]
            ax.bar(xs + (i - (len(names) - 1) / 2) * w, vals, w, label=student_name,
                   color=STUDENT_COLORS.get(student_name, OKABE_ITO[i]), edgecolor="black", linewidth=1.0)
        if not tch.empty:
            ax.axhline(tch["test_f1_macro"].max(), color=C_REF, ls=":", lw=2.2, label="Best teacher")
        ax.set_xticks(xs)
        ax.set_xticklabels(configs, rotation=25, ha="right")
        ax.set_ylabel("Test macro F1")
        ax.set_title("Which teacher configuration helps which student")
        ax.grid(True, axis="y", alpha=0.3, linestyle="--")
        ax.set_axisbelow(True)
        ax.legend(fontsize=11, loc="lower right")
        save_fig(fig, FIG_SUM / "teacher_vs_student")

        # 4) training time ---------------------------------------------------
        fig, ax = plt.subplots(figsize=(max(9.0, 0.7 * len(st)), 5.4))
        st2 = st.sort_values("fit_time_min")
        ax.bar(np.arange(len(st2)), st2["fit_time_min"], 0.65,
               color=[STUDENT_COLORS.get(s, "#7F7F7F") for s in st2["student"]], edgecolor="black", linewidth=1.0)
        ax.set_xticks(np.arange(len(st2)))
        ax.set_xticklabels(st2["model"], rotation=40, ha="right", fontsize=9)
        ax.set_ylabel("Training time (min)")
        ax.set_title("Distillation cost per configuration")
        ax.grid(True, axis="y", alpha=0.3, linestyle="--")
        ax.set_axisbelow(True)
        save_fig(fig, FIG_SUM / "training_time")

        # 5) validation loss curves -----------------------------------------
        fig, ax = plt.subplots(figsize=(9.6, 6.0))
        styles = ["-", "--", "-.", ":", (0, (5, 1)), (0, (3, 1, 1, 1)), (0, (1, 1)), (0, (5, 5))]
        for i, (_, r) in enumerate(st.iterrows()):
            h = OUT_DIR / r["student"] / r["config"] / "history.csv"
            if h.exists():
                hist = pd.read_csv(h)
                ax.plot(hist["epoch"], hist["val_loss"], lw=2.0,
                        color=STUDENT_COLORS.get(r["student"], "#7F7F7F"),
                        ls=styles[i % len(styles)], label=f"{r['student']} [{r['config']}]")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Validation loss (hard-label CE)")
        ax.set_title("Validation loss")
        ax.grid(True, alpha=0.3, linestyle="--")
        ax.set_axisbelow(True)
        ax.legend(ncol=2, fontsize=8, loc="upper right")
        save_fig(fig, FIG_SUM / "val_loss_all")

### 9.4 Reading the results
- **`none` is the control.** A configuration only earns its place if it beats that student's own `none` run; the "KD gain" column in 9.2 is exactly that difference.
- **The fused-teacher row** is the ensemble's own test score — the ceiling the students are distilled towards, at the cost of running all three networks.
- **Performance vs size** is the headline figure: teachers at 27–57M, students at 2.6–8.1M, and how much accuracy survives the compression.
- **Inference time** is measured on this machine at batch 32 after a warm-up, so it is comparable between models but not an absolute deployment figure.
- Differences below roughly 1.5 points of macro F1 are inside the noise of a 466-image test set.